In [ ]:
import os, subprocess, asyncio
from contextlib import AsyncExitStack
from dotenv import load_dotenv
from agents import Agent, Runner, Tool, trace
from agents.mcp import MCPServerStdio
from pathlib import Path
from datetime import datetime

load_dotenv(override=True)

# Import modules
from mcp_params_shop import nhan_vien_mcp_server_params, nghien_cuu_mcp_server_params
from templates_shop import (
    nghien_cuu_instructions, cong_cu_nghien_cuu,
    nhan_vien_instructions, nhiem_vu_phan_tich_doanh_thu
)

In [3]:
# Chuẩn bị dữ liệu mẫu để agent có gì mà làm việc
import time
from donhang import QuanLyDonHang

# Reset và tạo dữ liệu mới
if Path("donhang.db").exists():
    Path("donhang.db").unlink()

orders = [
    ("Nguyen Van A",  "Áo thun oversize",   3, 180_000, "Size L, giao buổi sáng"),
    ("Tran Thi B",    "Quần jean skinny",   1, 420_000, ""),
    ("Le Van C",      "Giày sneaker nam",   1, 950_000, "Size 42, hàng chính hãng"),
    ("Pham Thi D",    "Váy maxi hoa nhí",   2, 280_000, "Khách VIP, ưu tiên giao"),
    ("Hoang Van E",   "Áo khoác bomber",    1, 650_000, "Giao COD"),
    ("Vo Thi F",      "Túi tote canvas",    3, 150_000, ""),
]

ma_dons = []
for khach, sp, sl, gia, chu in orders:
    d = QuanLyDonHang.tao_don(khach, sp, sl, gia, chu)
    ma_dons.append(d["ma_don"])
    time.sleep(1)  # Đảm bảo timestamp unique

# Cập nhật một số trạng thái
QuanLyDonHang.cap_nhat_trang_thai(ma_dons[2], "dang_xu_ly")
QuanLyDonHang.cap_nhat_trang_thai(ma_dons[3], "da_giao")

print(f"Đã tạo {len(ma_dons)} đơn hàng mẫu")
print(f"Doanh thu: {QuanLyDonHang.doanh_thu()}")
print(f"Danh sách: {[d['trang_thai'] for d in QuanLyDonHang.danh_sach_don()]}")

Đã tạo 6 đơn hàng mẫu
Doanh thu: {'tong_doanh_thu': 3570000.0, 'so_don_hang': 6}
Danh sách: ['cho_xac_nhan', 'cho_xac_nhan', 'da_giao', 'dang_xu_ly', 'cho_xac_nhan', 'cho_xac_nhan']


In [5]:
# Test researcher agent
nc_params = nghien_cuu_mcp_server_params("test")

async with AsyncExitStack() as stack:
    nc_servers = [
        await stack.enter_async_context(
            MCPServerStdio(params=p, client_session_timeout_seconds=30)
        )
        for p in nc_params
    ]

    researcher = Agent(
        name="Researcher-Test",
        instructions=nghien_cuu_instructions(),
        model="gpt-4o-mini",
        mcp_servers=nc_servers
    )

    with trace("test-researcher"):
        r = await Runner.run(
            researcher,
            "Tìm kiếm xu hướng thời trang đang hot nhất tại Việt Nam tháng này."
            "Tóm tắt 3 xu hướng chính và cơ hội kinh doanh.",
            max_turns=15
        )
print(r.final_output)


### Tóm tắt xu hướng thời trang hot nhất tại Việt Nam tháng 4/2026

#### 1. **Áo Blouse Nữ Tính**
- **Đặc điểm**: Thiết kế với tay phồng, cổ vuông và chi tiết nơ, mang lại sự mềm mại và nữ tính.
- **Chất liệu**: Sử dụng cotton, voan hoặc lụa để đảm bảo sự thoáng mát trong mùa hè.
- **Màu sắc**: Bảng màu thiên về pastel và họa tiết hoa nhỏ, dễ dàng phối hợp với các trang phục khác.
- **Cơ hội kinh doanh**: Cung cấp nhiều mẫu mã và kích cỡ để phục vụ nhu cầu đi làm và đi chơi của phái nữ, tăng tính ứng dụng.

#### 2. **Quần Jeans Ống Suông**
- **Đặc điểm**: Phom dáng ống suông, thoải mái và che khuyết điểm hiệu quả.
- **Phối đồ**: Dễ dàng kết hợp với áo phông basic, áo sơ mi oversized hoặc áo hai dây.
- **Chất liệu**: Các phiên bản denim nhẹ hơn, dễ mặc hơn.
- **Cơ hội kinh doanh**: Xu hướng này có thể thu hút nhiều đối tượng, từ giới trẻ đến người trưởng thành nhờ tính đa dụng.

#### 3. **Quần Âu Ống Suông**
- **Đặc điểm**: Là món đồ không thể thiếu trong tủ đồ mùa hè, thiết kế ống thẳn

In [7]:
# Test Agent CSKH
nv_params = nhan_vien_mcp_server_params
nc_params = nghien_cuu_mcp_server_params("An")

async with AsyncExitStack() as stack:
    tat_ca_servers = [
        await stack.enter_async_context(
            MCPServerStdio(params=p, client_session_timeout_seconds=30)
        )
        for p in (nv_params + nc_params)
    ]
    nv_servers = tat_ca_servers[:len(nv_params)]
    nc_servers = tat_ca_servers[len(nv_params):]

    # Tao research tool
    researcher = Agent(
        name="Researcher-An",
        instructions=nghien_cuu_instructions(),
        model="gpt-4o-mini",
        mcp_servers=nc_servers
    )
    researcher_tool = researcher.as_tool(
        tool_name="researcher",
        tool_description=cong_cu_nghien_cuu()
    )

    # Tao nhan vien An
    an = Agent(
        name="An",
        instructions=nhan_vien_instructions("An", "Nhân viên CSKH"),
        tools=[researcher_tool],
        mcp_servers=nv_servers,
        model="gpt-4o-mini"
    )

    with trace("nhan-vien-an-phan-tich"):
        result = await Runner.run(
            an,
            nhiem_vu_phan_tich_doanh_thu("An"),
            max_turns=30
        )
print(result.final_output)

Đã hoàn thành các công việc phân tích cuối ngày như sau:

1. **Báo cáo doanh thu**: Tổng doanh thu trong ngày là 3.570.000 VNĐ, với 6 đơn hàng.
2. **Tỷ giá**:
   - **USD**: Mua tiền mặt 26.111,00 VNĐ; Mua chuyển khoản 26.141,00 VNĐ; Bán 26.361,00 VNĐ.
   - **EUR**: Mua tiền mặt 29.939,33 VNĐ; Mua chuyển khoản 30.241,75 VNĐ; Bán 31.517,66 VNĐ.
3. **Xu hướng thời trang** đã nghiên cứu:
   - Họa tiết hoa nền tối và váy ren màu đen rất được ưa chuộng.
   - Khuyến khích mua sắm từ các thương hiệu địa phương giúp tăng trưởng bền vững.
4. **Xuất báo cáo**: Tài liệu đã được lưu tại `sandbox/phan_tich_thi_truong_20260408.md`.

Đã gửi thông báo tóm tắt hoàn tất báo cáo. Nếu cần thêm thông tin, hãy cho tôi biết!


In [8]:
# kiem tra file bao cao
from pathlib import Path
from datetime import date

sandbox = Path("sandbox")
print("=== Files được tạo bởi agent ===\n")
if sandbox.exists():
    for f in sorted(sandbox.iterdir()):
        print(f"  {f.name}  ({f.stat().st_size:,} bytes)")
        if f.suffix == ".md" and f.stat().st_size < 3000:
            print(f.read_text(encoding="utf-8"))
            print()

print("\n=== Log thông báo ===")
import sqlite3
if Path("thong_bao.db").exists():
    conn = sqlite3.connect("thong_bao.db")
    rows = conn.execute(
        "SELECT ten_agent, loai, noi_dung, tao_luc FROM log_thong_bao ORDER BY id DESC LIMIT 5"
    ).fetchall()
    conn.close()
    for r in rows:
        icon = {"hoan_thanh": "✓", "bao_cao": "📊", "canh_bao": "⚠"}.get(r[1], "•")
        print(f"  {icon} [{r[0]}] {r[2]}")
        print(f"    {r[3][:19]}")
else:
    print("  Chưa có log (agent chưa gửi thông báo)")

=== Files được tạo bởi agent ===

  phan_tich_thi_truong_20260408.md  (1,283 bytes)
## Báo cáo Phân tích Thị trường

**Ngày lập:** 08/04/2026

### 1. Doanh thu
- **Tổng doanh thu trong ngày:** 3.570.000 VNĐ
- **Số đơn hàng:** 6

### 2. Bối cảnh Tài chính Vĩ mô
- **Tỷ giá USD:**
  - Mua tiền mặt: 26.111,00 VNĐ
  - Mua chuyển khoản: 26.141,00 VNĐ
  - Bán: 26.361,00 VNĐ

- **Tỷ giá EUR:**
  - Mua tiền mặt: 29.939,33 VNĐ
  - Mua chuyển khoản: 30.241,75 VNĐ
  - Bán: 31.517,66 VNĐ

- **Giá vàng hôm nay:** *Chưa cập nhật*

### 3. Xu hướng Thị trường từ Nghiên cứu
- **Họa tiết hoa nền tối** và **họa tiết khăn quàng** đang thịnh hành.
- **Váy ren** màu đen kết hợp với áo khoác Napoleon đang thu hút sự chú ý.
- **Phong cách thể thao nữ tính** với vải thể thao kết hợp thiết kế tinh tế được yêu thích.

#### Xu hướng tiêu dùng bền vững:
- Mua sắm từ các local brand và tiện ích thân thiện với môi trường đang gia tăng.

### 4. Khuyến nghị cho Chiến lược Bán hàng
- Tập trung vào các sản phẩm bền vững 

In [ ]:
from nhan_vien import NhanVien

# Tao nhan vien an
an = NhanVien("An", chuc_vu="Nhân viên CSKH")

await an.run()

In [ ]:
from nhan_vien import NhanVien, chay_tat_ca

doi_ngu = [
    NhanVien("An",   chuc_vu="Nhân viên CSKH"),
    NhanVien("Binh", chuc_vu="Chuyên viên Phân tích"),
    NhanVien("Chi",  chuc_vu="Chuyên viên Vận hành"),
    NhanVien("Dung", chuc_vu="Giám sát Hệ thống"),
]

ket_qua = await chay_tat_ca(doi_ngu)
for nv, kq in zip(doi_ngu, ket_qua):
    print(f"\n{nv.ten} ({nv.chuc_vu}):")
    print(f" {kq[:200]}..")

In [3]:
# Tong so tools

from mcp_params_shop import nhan_vien_mcp_server_params, nghien_cuu_mcp_server_params

tat_ca_params = nhan_vien_mcp_server_params + nghien_cuu_mcp_server_params("test")

tong_tools = 0
for p in tat_ca_params:
    async with MCPServerStdio(params=p, client_session_timeout_seconds=30) as server:
        tools = await server.list_tools()
        server_name = p["args"][-1].split("/")[-1].replace(".js", "").replace(".py", "")
        print(f"  {server_name}: {len(tools)} tools")
        tong_tools += len(tools)

print(f"\nTổng: {len(tat_ca_params)} MCP servers, {tong_tools} tools")
print("Mỗi nhân viên có quyền truy cập vào bộ tools này khi làm việc.")

  donhang_server: 5 tools
  dulieu_vn_server: 7 tools
  thong_bao_server: 2 tools
  sandbox: 14 tools
  index: 5 tools
  index: 9 tools

Tổng: 6 MCP servers, 42 tools
Mỗi nhân viên có quyền truy cập vào bộ tools này khi làm việc.
